In [3]:
import pandas as pd

In [4]:
data_clean_all = pd.read_csv("/kaggle/input/datasets/ssug4b/buat-klasifikasi/Berkas Untuk Klasifikasi All/Data part 1 Clean.csv")
data_clean_all.head()

,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful
0,rw5704482,raeldor-96879,After Life (2019– ),9.0,Very Strong Season 2,3 May 2020,0,"I enjoyed the first season, but I must say I t...","['1', '1']"
1,rw5704483,dosleeb,The Valhalla Murders (2019– ),6.0,Icelandic detectives?,3 May 2020,0,I know Iceland is a small country and police d...,"['2', '2']"
2,rw5704484,brightconscious,Special OPS (2020– ),7.0,Nothing special,3 May 2020,0,"Except K K , no other actor looks comfortable ...","['0', '0']"
3,rw5704485,gasconyway,#BlackAF (2020– ),8.0,Good but,3 May 2020,0,I'm guessing that as a 62 year old white woman...,"['5', '9']"
4,rw5704487,mmason-15867,The Droving (2020),2.0,An honest review,3 May 2020,0,Here's the truth. There's not much to this mov...,"['26', '41']"


In [5]:
data_berlabel = pd.read_csv("/kaggle/input/datasets/ssug4b/buat-klasifikasi/Berkas Untuk Klasifikasi All/data_sampling_labelbagging.csv")
data_berlabel.head()

,review_id,movie,rating,review_detail,label_komen
0,rw5704482,After Life (2019– ),9.0,"I enjoyed the first season, but I must say I t...",1
1,rw5704483,The Valhalla Murders (2019– ),6.0,I know Iceland is a small country and police d...,0
2,rw5704484,Special OPS (2020– ),7.0,"Except K K , no other actor looks comfortable ...",0
3,rw5704485,#BlackAF (2020– ),8.0,I'm guessing that as a 62 year old white woman...,1
4,rw5704487,The Droving (2020),2.0,Here's the truth. There's not much to this mov...,0


In [6]:
dict_berlabel = {}
for i in range(data_berlabel.shape[0]) :
    dict_berlabel[data_berlabel["review_id"][i]] = data_berlabel["label_komen"][i]

for i in dict_berlabel :
    print(i,dict_berlabel[i])
    break

rw5704482 1


In [7]:
dict_klasifikasi = {"review_id" : [], "review_detail" : []}

for i in range(data_clean_all.shape[0]) :
    if data_clean_all["review_id"][i] not in dict_berlabel :
        dict_klasifikasi["review_id"].append(data_clean_all["review_id"][i])
        dict_klasifikasi["review_detail"].append(data_clean_all["review_detail"][i])
        
df_klasifikasi = pd.DataFrame(dict_klasifikasi)
df_klasifikasi.head()

,review_id,review_detail
0,rw5704548,I purchased the movie on Amazon so I have it d...
1,rw5704928,"A very, very bad episode and out of content th..."
2,rw5705227,Maybe this show should call Lex Luthor and no ...
3,rw5705231,It's very good and it's always good to see a f...
4,rw5705299,"Unlike some others who have commented here, I ..."


In [8]:
import pandas as pd
import re
import nltk
from IPython.display import display

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [9]:
def case_folding(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_klasifikasi['case_folded'] = df_klasifikasi['review_detail'].apply(case_folding)

display(df_klasifikasi[['review_detail', 'case_folded']].head(3))

,review_detail,case_folded
0,I purchased the movie on Amazon so I have it d...,i purchased the movie on amazon so i have it d...
1,"A very, very bad episode and out of content th...",a very very bad episode and out of content the...
2,Maybe this show should call Lex Luthor and no ...,maybe this show should call lex luthor and no ...


In [10]:
from nltk.tokenize import word_tokenize

df_klasifikasi['tokenized'] = df_klasifikasi['case_folded'].apply(word_tokenize)

display(df_klasifikasi[['case_folded', 'tokenized']].head(3))

,case_folded,tokenized
0,i purchased the movie on amazon so i have it d...,"[i, purchased, the, movie, on, amazon, so, i, ..."
1,a very very bad episode and out of content the...,"[a, very, very, bad, episode, and, out, of, co..."
2,maybe this show should call lex luthor and no ...,"[maybe, this, show, should, call, lex, luthor,..."


In [11]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df_klasifikasi['no_stopwords'] = df_klasifikasi['tokenized'].apply(remove_stopwords)

display(df_klasifikasi[['tokenized', 'no_stopwords']].head(3))

,tokenized,no_stopwords
0,"[i, purchased, the, movie, on, amazon, so, i, ...","[purchased, movie, amazon, digitally, buy, dvd..."
1,"[a, very, very, bad, episode, and, out, of, co...","[bad, episode, content, main, character, episode]"
2,"[maybe, this, show, should, call, lex, luthor,...","[maybe, show, call, lex, luthor, supergirl]"


In [12]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df_klasifikasi['lemmatized'] = df_klasifikasi['no_stopwords'].apply(lemmatize_text)

df_klasifikasi['clean_review'] = df_klasifikasi['lemmatized'].apply(lambda x: ' '.join(x))

display(df_klasifikasi[['no_stopwords', 'clean_review']].head(3))

,no_stopwords,clean_review
0,"[purchased, movie, amazon, digitally, buy, dvd...",purchased movie amazon digitally buy dvd watch...
1,"[bad, episode, content, main, character, episode]",bad episode content main character episode
2,"[maybe, show, call, lex, luthor, supergirl]",maybe show call lex luthor supergirl


In [13]:
del df_klasifikasi["review_detail"]
del df_klasifikasi["case_folded"]
del df_klasifikasi["tokenized"]
del df_klasifikasi["no_stopwords"]
del df_klasifikasi["lemmatized"]

In [14]:
df_klasifikasi.head()

,review_id,clean_review
0,rw5704548,purchased movie amazon digitally buy dvd watch...
1,rw5704928,bad episode content main character episode
2,rw5705227,maybe show call lex luthor supergirl
3,rw5705231,good always good see familiar face comedy hard...
4,rw5705299,unlike others commented think kept political b...


In [16]:
import pickle

def load_model(filepath):
    with open(filepath, 'rb') as file:
        return pickle.load(file)

tfidf = load_model('/kaggle/input/datasets/ssug4b/buat-klasifikasi/Berkas Untuk Klasifikasi All/Tf_Idf.pkl')
base_model_1 = load_model('/kaggle/input/datasets/ssug4b/buat-klasifikasi/Berkas Untuk Klasifikasi All/Naive_Bayes_B1.pkl')
base_model_2 = load_model('/kaggle/input/datasets/ssug4b/buat-klasifikasi/Berkas Untuk Klasifikasi All/Support_Vector_Machine_B2.pkl')
base_model_3 = load_model('/kaggle/input/datasets/ssug4b/buat-klasifikasi/Berkas Untuk Klasifikasi All/Random_Forest_B3.pkl')
meta_model = load_model('/kaggle/input/datasets/ssug4b/buat-klasifikasi/Berkas Untuk Klasifikasi All/Logistic_Linear_Meta.pkl')

In [18]:
import cudf
import cupy as cp


data_gpu = cudf.Series(df_klasifikasi["clean_review"])

X_tfidf = tfidf.transform(data_gpu)
X_tfidf = X_tfidf.toarray()
p1 = base_model_1.predict(X_tfidf)
p2 = base_model_2.predict(X_tfidf)
p3 = base_model_3.predict(X_tfidf)

X_meta = cudf.DataFrame({'p1': p1, 'p2': p2, 'p3': p3})

pred_gpu = meta_model.predict(X_meta)

if hasattr(pred_gpu, 'to_numpy'):
    pred_cpu = pred_gpu.to_numpy()
elif hasattr(pred_gpu, 'get'):
    pred_cpu = pred_gpu.get()
else:
    pred_cpu = pred_gpu
    
df_klasifikasi["label_komen"] = list(pred_cpu)

df_klasifikasi.head()

,review_id,clean_review,label_komen
0,rw5704548,purchased movie amazon digitally buy dvd watch...,1
1,rw5704928,bad episode content main character episode,0
2,rw5705227,maybe show call lex luthor supergirl,0
3,rw5705231,good always good see familiar face comedy hard...,1
4,rw5705299,unlike others commented think kept political b...,0


In [19]:
dict_k_berlabel = {}

for i in range(df_klasifikasi.shape[0]) :
    dict_k_berlabel[df_klasifikasi["review_id"][i]] = df_klasifikasi["label_komen"][i]

for i in dict_k_berlabel :
    print(i,dict_k_berlabel[i])
    break

rw5704548 1


In [20]:
label = []

for i in data_clean_all["review_id"] :
    if i in dict_berlabel :
        label.append(dict_berlabel[i])
    else :
        label.append(dict_k_berlabel[i])

data_clean_all["label_komen"] = label

data_clean_all.head()

,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful,label_komen
0,rw5704482,raeldor-96879,After Life (2019– ),9.0,Very Strong Season 2,3 May 2020,0,"I enjoyed the first season, but I must say I t...","['1', '1']",1
1,rw5704483,dosleeb,The Valhalla Murders (2019– ),6.0,Icelandic detectives?,3 May 2020,0,I know Iceland is a small country and police d...,"['2', '2']",0
2,rw5704484,brightconscious,Special OPS (2020– ),7.0,Nothing special,3 May 2020,0,"Except K K , no other actor looks comfortable ...","['0', '0']",0
3,rw5704485,gasconyway,#BlackAF (2020– ),8.0,Good but,3 May 2020,0,I'm guessing that as a 62 year old white woman...,"['5', '9']",1
4,rw5704487,mmason-15867,The Droving (2020),2.0,An honest review,3 May 2020,0,Here's the truth. There's not much to this mov...,"['26', '41']",0


In [21]:
data_clean_all.to_csv("Hasil_Klasifikasi_Review.csv", index= False)